In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_train_t.shape

torch.Size([168834, 53])

In [3]:
class IDSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [5]:
NUM_CLIENTS = 5

def partition_noniid(X, y, num_clients):
    # sorting by label so attacks and normal traffic aren't mixed randomly
    sorted_idx = np.argsort(y.flatten())
    X_sorted, y_sorted = X[sorted_idx], y[sorted_idx]

    # splitting the sorted data unevenly — each client gets a different "slice"
    # of the label spectrum, so some end up mostly normal, others mostly attack
    splits = np.array_split(np.arange(len(X_sorted)), num_clients)
    return [(X_sorted[s], y_sorted[s]) for s in splits]

client_data = partition_noniid(X_train_t.numpy(), y_train_t.numpy(), NUM_CLIENTS)

# checking how skewed each client's data actually is
for i, (X, y) in enumerate(client_data):
    print(f"Client {i}: {len(y)} samples, {y.mean():.2%} attack")

Client 0: 33767 samples, 0.00% attack
Client 1: 33767 samples, 81.60% attack
Client 2: 33767 samples, 100.00% attack
Client 3: 33767 samples, 100.00% attack
Client 4: 33766 samples, 100.00% attack


In [7]:
from flwr.client import NumPyClient, ClientApp
from flwr.common import Context
from collections import OrderedDict

class FlowerClient(NumPyClient):
    def __init__(self, model, X, y):
        self.model = model
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def _get_params(self):
        return [val.cpu().numpy() for val in self.model.state_dict().values()]

    def _set_params(self, params):
        params_dict = zip(self.model.state_dict().keys(), params)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def get_parameters(self, config):
        return self._get_params()

    def fit(self, parameters, config):
        self._set_params(parameters)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.model.train()
        for _ in range(3):
            optimizer.zero_grad()
            outputs = self.model(self.X)
            loss = criterion(outputs, self.y)
            loss.backward()
            optimizer.step()
        return self._get_params(), len(self.X), {}

    def evaluate(self, parameters, config):
        self._set_params(parameters)
        criterion = nn.BCELoss()
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(self.X)
            loss = criterion(outputs, self.y).item()
            preds = (outputs > 0.5).float()
            acc = (preds == self.y).float().mean().item()
        return loss, len(self.X), {"accuracy": acc}

In [9]:
def client_fn(context: Context):
    partition_id = int(context.node_config["partition-id"])
    X, y = client_data[partition_id]
    model = IDSNet(input_dim=X.shape[1])
    return FlowerClient(model, X, y).to_client()

client_app = ClientApp(client_fn=client_fn)

In [11]:
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg

def weighted_average(metrics):
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}

def server_fn(context: Context):
    strategy = FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_available_clients=NUM_CLIENTS,
        evaluate_metrics_aggregation_fn=weighted_average,
    )
    config = ServerConfig(num_rounds=15)
    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

In [13]:
from flwr.simulation import run_simulation

run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
    backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 0}},
)

C:\Users\Admin\anaconda3\envs\fl-ids\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-23 13:21:55,814	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower ServerApp, config: num_rounds=15, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
C:\Users\Admin\anaconda3\envs\fl-ids\lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn of

In [15]:
noniid_results = {
    "final_accuracy": 0.76325,
    "rounds": 15,
    "client_skew": [0.0, 0.816, 1.0, 1.0, 1.0],
    "note": "accuracy plateaued near the overall attack rate (~76%), suggesting the model collapsed toward predicting the majority class under extreme non-IID conditions"
}
import json
with open('federated_noniid_results.json', 'w') as f:
    json.dump(noniid_results, f)